# 11 — Cross-site validation

**Purpose.** UCI heart-disease sites as natural clients: within-site holdout (protocol A) and leave-one-site-out external validation (protocol B).

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/11_cross_site_validation_*.png|pdf` and `reports/notebooks/11_cross_site_validation.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "11_cross_site_validation"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Protocol A and B

In [2]:
from federated_learning.preprocessing import load_sites, load_loso, SITE_NAMES, site_validation
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
cfg = FLConfig(rounds=25, local_epochs=2, hidden=16, seed=SEED)
A_ = load_sites(SEED); ra = run_fl(cfg, A_, A_.parts); print("Protocol A (within-site holdout) acc:", round(ra.final["acc"], 3), "| sites:", A_.site_names)
rows = []
for s in SITE_NAMES:
    L = load_loso(s); r = run_fl(cfg, L, L.parts); rows.append(dict(held_out_site=s, n_test=len(L.y_test), acc=r.final["acc"], auc=r.final["auc"], test_positive_rate=float(L.y_test.mean())))
tab = pd.DataFrame(rows).round(3); display(tab); SUMMARY["loso"] = tab.to_dict("records")

Protocol A (within-site holdout) acc: 0.791 | sites: ('cleveland', 'hungary', 'switzerland', 'va_long_beach')


,held_out_site,n_test,acc,auc,test_positive_rate
0,cleveland,303,0.759,0.858,0.459
1,hungary,294,0.789,0.885,0.361
2,switzerland,123,0.715,0.714,0.935
3,va_long_beach,200,0.730,0.733,0.745


## 2. Figure

In [3]:
fig, ax = plt.subplots(figsize=(5, 2.9)); ax.bar(tab.held_out_site, tab.acc, color=PAL[0]); ax.set_ylabel("accuracy on held-out site"); ax.set_title("Leave-one-site-out (small, 1980s cohorts)"); plt.setp(ax.get_xticklabels(), rotation=15)
save_fig(fig, "loso"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/11_cross_site_validation.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.